# CS336 Assignment 1 (basics): Building a Transformer LM


## Byte-Pair Encoding (BPE) Tokenizer


### The Unicode Standard


In [3]:
ord("牛")

29275

In [4]:
chr(29275)

'牛'

(a) What Unicode character does chr(0) return?


In [5]:
chr(0)

'\x00'

(b) How does this character’s string representation (**repr**()) differ from its printed
representation?

- character's string representation is the unambiguous, technical string
- printed representation is human readable, user-friendly output


In [6]:
print(chr(0))

 


In [7]:
len(repr(chr(0)))

6

In [8]:
len(chr(0))

1

In [9]:
len("this is a test" + chr(0) + "string")

21

In [10]:
len("this is a teststring")

20

(c) What happens when this character occurs in text?

- The null character is embedded invisibly in the string—it's present and counted in the length, but displays as nothing when printed.


In [11]:
"this is a test" + chr(0) + "string"

'this is a test\x00string'

In [12]:
print("this is a test" + chr(0) + "string")

this is a test string


### Unicode Encodings


In [13]:
test_string = "hello! こんにちは!"
utf8_encoded = test_string.encode("utf-8")
print(utf8_encoded)

b'hello! \xe3\x81\x93\xe3\x82\x93\xe3\x81\xab\xe3\x81\xa1\xe3\x81\xaf!'


In [14]:
print(type(utf8_encoded))

<class 'bytes'>


In [15]:
# Get the byte values for the encoded string (integers from 0 to 255).
list(utf8_encoded)

[104,
 101,
 108,
 108,
 111,
 33,
 32,
 227,
 129,
 147,
 227,
 130,
 147,
 227,
 129,
 171,
 227,
 129,
 161,
 227,
 129,
 175,
 33]

In [16]:
# One byte does not necessarily correspond to one Unicode character!
print(len(test_string))

13


In [17]:
print(len(utf8_encoded))

23


In [18]:
print(utf8_encoded.decode("utf-8"))

hello! こんにちは!


(a) What are some reasons to prefer training our tokenizer on UTF-8 encoded bytes, rather than
UTF-16 or UTF-32?

- UTF-8 is preferred because it is the most compact encoding for ASCII-heavy text (1 byte per character vs. 2 or 4), resulting in a smaller base vocabulary (256 byte values vs. 65,536+ for UTF-16/UTF-32), which makes tokenizer training more efficient.


(b) Consider the following (incorrect) function, which is intended to decode a UTF-8 byte string
into a Unicode string. Why is this function incorrect? Provide an example of an input byte
string that yields incorrect results.

- It fails because the function decodes each byte individually, but multi-byte UTF-8 characters (like é, which is 2 bytes) cannot be decoded as separate single bytes.


In [ ]:
def decode_utf8_bytes_to_str_wrong(bytestring: bytes):
    return "".join([bytes([b]).decode("utf-8") for b in bytestring])


decode_utf8_bytes_to_str_wrong("hello".encode("utf-8"))

In [ ]:
decode_utf8_bytes_to_str_wrong("é".encode("utf-8"))

(c) Give a two-byte sequence that does not decode to any Unicode character(s).

- b'\xc0\x00' is invalid UTF-8 because 0xc0 signals a two-byte sequence but 0x00 is not a valid continuation byte (continuation bytes must be in the range 0x80–0xBF).


### Subword Tokenization


### BPE Tokenization Training


In [ ]:
PAT = r"""'(?:[sdmt]|ll|ve|re)| ?\p{L}+| ?\p{N}+| ?[^\s\p{L}\p{N}]+|\s+(?!\S)|\s+"""

In [ ]:
import regex as re

re.findall(PAT, "some text that i'll pre-tokenize")

In [ ]:
from cs336_basics.tokenizer import train_bpe

import os

path = os.path.join("..", "tests/fixtures", "tinystories_sample.txt")

print(path)

vocab, merges = train_bpe(path, vocab_size=266, verbose=True)

In [1]:
import os
import time
import pickle
import resource

from cs336_basics.tokenizer import train_bpe

import cProfile
import pstats

profiler = cProfile.Profile()
profiler.enable()


path = os.path.join("..", "data", "TinyStoriesV2-GPT4-train.txt")
out_path = os.path.join("..", "outputs", "TinyStoriesV2-GPT4-train-10K.pkl")
os.makedirs(os.path.dirname(out_path), exist_ok=True)

t0 = time.perf_counter()
vocab, merges = train_bpe(path, vocab_size=10000, special_tokens=["<|endoftext|>"])
elapsed = time.perf_counter() - t0

profiler.disable()
stats = pstats.Stats(profiler)
stats.sort_stats("cumulative")
stats.print_stats("tokenizer.py")

print(f"Training time:     {elapsed / 60:.2f} min")
print(f"Peak memory main:  {resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / (1024**2):.0f} MB")
print(f"Peak memory child: {resource.getrusage(resource.RUSAGE_CHILDREN).ru_maxrss / (1024**2):.0f} MB")

with open(out_path, "wb") as f:
    pickle.dump({"vocab": vocab, "merges": merges}, f)

longest = max(vocab.values(), key=len)
print(f"Longest token: {longest!r} ({len(longest)} bytes)")

         24518225 function calls (24518127 primitive calls) in 77.116 seconds

   Ordered by: cumulative time
   List reduced from 490 to 8 due to restriction <'tokenizer.py'>

   ncalls  tottime  percall  cumtime  percall filename:lineno(function)
        1    0.000    0.000   66.480   66.480 /Volumes/Projects/Github/Courses/Stanford CS336 - Language Modeling From Scratch/cs336-assignment1-basics/cs336_basics/tokenizer.py:224(pretokenize)
        1    0.122    0.122    0.354    0.354 /Volumes/Projects/Github/Courses/Stanford CS336 - Language Modeling From Scratch/cs336-assignment1-basics/cs336_basics/tokenizer.py:276(train_bpe)
  2037058    0.111    0.000    0.111    0.000 /Volumes/Projects/Github/Courses/Stanford CS336 - Language Modeling From Scratch/cs336-assignment1-basics/cs336_basics/tokenizer.py:15(__init__)
   769991    0.047    0.000    0.047    0.000 /Volumes/Projects/Github/Courses/Stanford CS336 - Language Modeling From Scratch/cs336-assignment1-basics/cs336_basics/tokeniz

In [2]:
import os
import time
import pickle
import resource

from cs336_basics.tokenizer import train_bpe

import cProfile
import pstats

profiler = cProfile.Profile()
profiler.enable()


path = os.path.join("..", "data", "owt_train.txt")
out_path = os.path.join("..", "outputs", "owt_train_32K.pkl")
os.makedirs(os.path.dirname(out_path), exist_ok=True)

t0 = time.perf_counter()
vocab, merges = train_bpe(path, vocab_size=32000, special_tokens=["<|endoftext|>"])
elapsed = time.perf_counter() - t0

profiler.disable()
stats = pstats.Stats(profiler)
stats.sort_stats("cumulative")
stats.print_stats("tokenizer.py")

print(f"Training time:     {elapsed / 60:.2f} min")
print(f"Peak memory main:  {resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / (1024**2):.0f} MB")
print(f"Peak memory child: {resource.getrusage(resource.RUSAGE_CHILDREN).ru_maxrss / (1024**2):.0f} MB")

with open(out_path, "wb") as f:
    pickle.dump({"vocab": vocab, "merges": merges}, f)

longest = max(vocab.values(), key=len)
print(f"Longest token: {longest!r} ({len(longest)} bytes)")

         4679190878 function calls (4679190642 primitive calls) in 3681.481 seconds

   Ordered by: cumulative time
   List reduced from 392 to 8 due to restriction <'tokenizer.py'>

   ncalls  tottime  percall  cumtime  percall filename:lineno(function)
        1    0.000    0.000  418.770  418.770 /Volumes/Projects/Github/Courses/Stanford CS336 - Language Modeling From Scratch/cs336-assignment1-basics/cs336_basics/tokenizer.py:224(pretokenize)
433233053   25.708    0.000   25.708    0.000 /Volumes/Projects/Github/Courses/Stanford CS336 - Language Modeling From Scratch/cs336-assignment1-basics/cs336_basics/tokenizer.py:21(__eq__)
397808119   20.004    0.000   20.004    0.000 /Volumes/Projects/Github/Courses/Stanford CS336 - Language Modeling From Scratch/cs336-assignment1-basics/cs336_basics/tokenizer.py:18(__lt__)
298965176   16.065    0.000   16.065    0.000 /Volumes/Projects/Github/Courses/Stanford CS336 - Language Modeling From Scratch/cs336-assignment1-basics/cs336_basics/tokeniz

In [34]:
from cs336_basics.tokenizer import Tokenizer

import os
import pickle

In [35]:
tok_path = os.path.join("..", "outputs", "owt_train_32K.pkl")
with open(tok_path, "rb") as f:
    data = pickle.load(f)

vocab = data["vocab"]
merges = data["merges"]

owt_tokenizer = Tokenizer(vocab, merges, special_tokens=["<|endoftext|>"])


In [42]:
data_path = "../data/owt_valid.txt"

with open(data_path) as f:
    contents = f.read()

# Sample 10 docs
owt_docs = contents.split("<|endoftext|>")[:10]

ids = []
num_bytes = 0

for doc in owt_docs:
    ids.extend(owt_tokenizer.encode(doc))
    num_bytes += len(bytes(doc, encoding="UTF-8"))

print(num_bytes / len(ids))

4.504954914739756


In [37]:
tok_path = os.path.join("..", "outputs", "TinyStoriesV2-GPT4-train-10K.pkl")
with open(tok_path, "rb") as f:
    data = pickle.load(f)

vocab = data["vocab"]
merges = data["merges"]

ts_tokenizer = Tokenizer(vocab, merges, special_tokens=["<|endoftext|>"])

In [41]:
data_path = "../data/TinyStoriesV2-GPT4-valid.txt"

with open(data_path) as f:
    contents = f.read()

# Sample 10 docs
ts_docs = contents.split("<|endoftext|>")[:10]

ids = []
num_bytes = 0

for doc in ts_docs:
    ids.extend(ts_tokenizer.encode(doc))
    num_bytes += len(bytes(doc, encoding="UTF-8"))

print(num_bytes / len(ids))

4.011164867882397


In [ ]:
ids = []
num_bytes = 0

for doc in owt_docs:
    ids.extend(ts_tokenizer.encode(doc))
    num_bytes += len(bytes(doc, encoding="UTF-8"))

print(num_bytes / len(ids))

3.404628567573038


In [ ]:
data_path = "../data/TinyStoriesV2-GPT4-valid.txt"
file_size = os.path.getsize(data_path)  # in bytes
ct = 0
with open(data_path) as f:
    token_stream = ts_tokenizer.encode_iterable(f)

    start_time = time.time()
    while next(token_stream, -1) != -1:
        ct += 1
    end_time = time.time()

total_time = end_time - start_time
bytes_per_second = file_size / total_time

print(f"{ct:,} tokens processed")
print(f"{file_size:,} bytes processed")
print(f"{bytes_per_second:,.2f} bytes/s")
print(f"{ct / total_time:,.2f} tok/s")


5,461,210 tokens processed
22,502,601 bytes processed
7,256,632.85 bytes/s
1,761,129.57 tok/s


In [49]:
pile_data_size = 8.25e11
time_to_process = pile_data_size / bytes_per_second
print(f"Estimated time to process Pile dataset: {time_to_process / 3600:.2f} hours")

Estimated time to process Pile dataset: 31.58 hours


In [50]:
import os
import numpy as np

data, outs = os.path.join("..", "data"), os.path.join("..", "outputs")

jobs = [
    (ts_tokenizer, "TinyStoriesV2-GPT4-train.txt", "ts_train.npy"),
    (ts_tokenizer, "TinyStoriesV2-GPT4-valid.txt", "ts_valid.npy"),
    (owt_tokenizer, "owt_train.txt", "owt_train.npy"),
    (owt_tokenizer, "owt_valid.txt", "owt_valid.npy"),
]

for tok, txt, out in jobs:
    with open(os.path.join(data, txt), encoding="utf-8") as f:
        ids = np.fromiter(tok.encode_iterable(f), dtype=np.uint16)
    np.save(os.path.join(outs, out), ids)
    print(f"{out}: {len(ids):,} tokens")

ts_train.npy: 540,796,778 tokens
ts_valid.npy: 5,461,210 tokens
owt_train.npy: 2,727,120,452 tokens
owt_valid.npy: 66,401,098 tokens
